# Аналіз E-Commerce платформи Olist

End-to-end аналітичний проєкт на основі реальних даних бразильського маркетплейсу **Olist** — платформи, що з'єднує малих продавців з великими торговими майданчиками Бразилії.

Проєкт охоплює повний цикл роботи з даними: проектування реляційної бази даних → ETL-пайплайн (Python → PostgreSQL) → SQL-аналіз → візуалізації → статистичне тестування гіпотез.         
Мета — не лише технічна реалізація, а й здатність формулювати бізнес-питання, знаходити в даних відповіді та перетворювати їх на конкретні рекомендації.

**Три ключових аналітичних питання проєкту:**

1. Що обмежує масштабування виручки: географічна концентрація пропозиції чи логістичні бар'єри у регіонах із вищим середнім чеком?
2. На якому пороговому значенні часу доставки задоволеність клієнтів різко падає — і чия відповідальність: продавця чи перевізника?
3. Які структурні причини утримують retention rate на рівні ~3% і які зміни дадуть найбільший ефект?

## Датасет

**Джерело:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle)  
**Період:** Вересень 2016 — Жовтень 2018  
**Обсяг:** ~100k замовлень · ~100k відгуків · ~3k продавців · ~32k товарів  
**Формат:** 9 CSV-файлів

| Файл | Що містить |
|---|---|
| `olist_orders_dataset.csv` | Замовлення, статуси, дати |
| `olist_order_items_dataset.csv` | Товари в замовленнях, ціна, фрахт |
| `olist_order_payments_dataset.csv` | Платежі, тип, сума, розстрочка |
| `olist_order_reviews_dataset.csv` | Відгуки клієнтів (оцінка 1–5) |
| `olist_products_dataset.csv` | Каталог товарів, категорія, розміри |
| `olist_sellers_dataset.csv` | Продавці, місто, штат |
| `olist_customers_dataset.csv` | Клієнти, місто, штат |
| `olist_geolocation_dataset.csv` | Геолокація за zip-кодом |
| `product_category_name_translation.csv` | Переклад категорій порт. → англ. |

## Аналітичні питання

1. Яка динаміка замовлень по місяцях — чи росте платформа?
2. Які топ-10 категорій генерують найбільший дохід?
3. Який середній час доставки по штатах і де найгірша логістика?
4. Як розподіляються оцінки відгуків і який % незадоволених клієнтів?
5. Хто топ-10 продавців за активністю і яка їхня середня оцінка?
6. Яка частка клієнтів повертається для повторної покупки?
7. Який середній чек по кожному типу оплати?
8. Чи впливає час доставки на оцінку клієнта?
9. Яка географія доходів платформи — які штати генерують найбільший GMV і яка середня цінність замовлення по регіонах?

## Технічний стек

| Інструмент | Для чого |
|---|---|
| `Python` + `pandas` | Очищення та трансформація даних |
| `SQLAlchemy` + `psycopg2` | Підключення та завантаження в PostgreSQL |
| `PostgreSQL` | Зберігання даних, SQL-аналіз |
| `matplotlib` + `seaborn` | Візуалізації |
| `scipy` | Статистичні тести |
| `Docker` | Локальний PostgreSQL-контейнер |
| `Jupyter Notebook` | Документування аналізу |

## Детальний опис файлів датасету

---

### `olist_orders_dataset.csv`
Центральна таблиця датасету. Кожен рядок — одне замовлення. До неї через `order_id` приєднуються всі інші таблиці: товари, платежі, відгуки.

| Колонка | Опис |
|---|---|
| `order_id` | Унікальний ідентифікатор замовлення (PK) |
| `customer_id` | ID клієнта, що зробив замовлення (FK → customers) |
| `order_status` | Статус замовлення: `delivered`, `shipped`, `canceled`, `processing` та ін. |
| `order_purchase_timestamp` | Дата і час оформлення замовлення |
| `order_approved_at` | Дата і час підтвердження оплати |
| `order_delivered_carrier_date` | Дата передачі замовлення перевізнику |
| `order_delivered_customer_date` | Фактична дата доставки клієнту (~3% NULL — замовлення ще в дорозі або скасовані) |
| `order_estimated_delivery_date` | Очікувана дата доставки, яку Olist обіцяє клієнту на момент оформлення |

---

### `olist_order_items_dataset.csv`
Товарний склад кожного замовлення. Одне замовлення може містити кілька позицій — тоді буде кілька рядків з однаковим `order_id`. Первинний ключ — складений: `(order_id, order_item_id)`.

| Колонка | Опис |
|---|---|
| `order_id` | ID замовлення (FK → orders) |
| `order_item_id` | Порядковий номер товару всередині замовлення (1, 2, 3…) |
| `product_id` | ID товару (FK → products) |
| `seller_id` | ID продавця, який продає цей товар (FK → sellers) |
| `shipping_limit_date` | Крайній термін, до якого продавець має передати товар перевізнику |
| `price` | Ціна товару в бразильських реалах (BRL) |
| `freight_value` | Вартість доставки для цього товару (BRL) |

---

### `olist_order_payments_dataset.csv`
Платіжні транзакції по замовленнях. Одне замовлення може мати кілька рядків — наприклад, якщо клієнт платив частково ваучером, а частково карткою. Первинний ключ — складений: `(order_id, payment_sequential)`.

| Колонка | Опис |
|---|---|
| `order_id` | ID замовлення (FK → orders) |
| `payment_sequential` | Порядковий номер платежу в рамках одного замовлення |
| `payment_type` | Тип оплати: `credit_card`, `boleto`, `voucher`, `debit_card` |
| `payment_installments` | Кількість розстрочок (для кредитної картки; 1 = без розстрочки) |
| `payment_value` | Сума транзакції в BRL |

---

### `olist_order_reviews_dataset.csv`
Відгуки клієнтів після отримання замовлення. Текстові поля заповнюються рідко: `review_comment_title` порожній у ~88% відгуків, `review_comment_message` — у ~59%. Це нормальна поведінка користувачів, порожні значення зберігаються як NULL.

| Колонка | Опис |
|---|---|
| `review_id` | Унікальний ідентифікатор відгуку (PK) |
| `order_id` | ID замовлення, до якого належить відгук (FK → orders) |
| `review_score` | Оцінка від 1 до 5 (1 — найгірша, 5 — найкраща) |
| `review_comment_title` | Заголовок відгуку (~88% NULL) |
| `review_comment_message` | Текст відгуку (~59% NULL) |
| `review_creation_date` | Дата, коли Olist надіслав клієнту запит на відгук |
| `review_answer_timestamp` | Дата і час, коли клієнт заповнив відгук |

---

### `olist_products_dataset.csv`
Каталог товарів, що продаються на платформі. Категорії зберігаються португальською — для аналізу використовується таблиця перекладів `product_category_name_translation.csv`.

> **Примітка:** колонки `product_name_lenght` і `product_description_lenght` містять помилку у назві (`lenght` замість `length`) — це оригінальна опечатка датасету Olist.

| Колонка | Опис |
|---|---|
| `product_id` | Унікальний ідентифікатор товару (PK) |
| `product_category_name` | Назва категорії товару португальською мовою |
| `product_name_lenght` | Кількість символів у назві товару *(typo в оригіналі)* |
| `product_description_lenght` | Кількість символів в описі товару *(typo в оригіналі)* |
| `product_photos_qty` | Кількість фотографій товару в картці |
| `product_weight_g` | Вага товару в грамах |
| `product_length_cm` | Довжина товару в сантиметрах |
| `product_height_cm` | Висота товару в сантиметрах |
| `product_width_cm` | Ширина товару в сантиметрах |

---

### `olist_sellers_dataset.csv`
Інформація про продавців, зареєстрованих на платформі.

| Колонка | Опис |
|---|---|
| `seller_id` | Унікальний ідентифікатор продавця (PK) |
| `seller_zip_code_prefix` | Перші 5 цифр поштового індексу продавця |
| `seller_city` | Місто продавця |
| `seller_state` | Штат продавця (двобуквений код, напр. `SP`, `RJ`) |

---

### `olist_customers_dataset.csv`
Інформація про клієнтів. Важлива особливість анонімізації: кожне замовлення отримує окремий `customer_id`, навіть якщо одна людина робила кілька замовлень. Щоб ідентифікувати реального клієнта і відстежити повторні покупки, потрібно використовувати `customer_unique_id`.

| Колонка | Опис |
|---|---|
| `customer_id` | ID клієнта в рамках конкретного замовлення (PK) — унікальний для кожного замовлення |
| `customer_unique_id` | Справжній унікальний ідентифікатор клієнта — однаковий для всіх замовлень однієї людини |
| `customer_zip_code_prefix` | Перші 5 цифр поштового індексу клієнта |
| `customer_city` | Місто клієнта |
| `customer_state` | Штат клієнта (двобуквений код) |

---

### `olist_geolocation_dataset.csv`
GPS-координати для бразильських поштових індексів. Один zip-код може мати кілька записів — різні вулиці одного поштового індексу. При аналізі агрегуємо медіаною: одна точка на zip-код.

> **Примітка:** ця таблиця не завантажується в основну схему БД — використовується як довідник при побудові географічних візуалізацій у кроці 03_visualizations.ipynb.

| Колонка | Опис |
|---|---|
| `geolocation_zip_code_prefix` | Перші 5 цифр поштового індексу |
| `geolocation_lat` | Широта (latitude) |
| `geolocation_lng` | Довгота (longitude) |
| `geolocation_city` | Назва міста |
| `geolocation_state` | Штат (двобуквений код) |

---

### `product_category_name_translation.csv`
Словник для перекладу категорій товарів з португальської на англійську (71 категорія). При ETL-трансформації приєднується до таблиці `products` через LEFT JOIN по полю `product_category_name` — щоб всі товари отримали англійську назву категорії для подальшого аналізу.

| Колонка | Опис |
|---|---|
| `product_category_name` | Назва категорії португальською (ключ для join з products) |
| `product_category_name_english` | Назва категорії англійською |

## Структура проєкту

Цей ноутбук — точка входу: опис датасету, аналітичних питань і технічного стеку. Подальший аналіз розгортається у такому порядку:

| Ноутбук | Зміст |
|---|---|
| `01_eda.ipynb` | Первинне дослідження даних: розміри таблиць, типи колонок, пропуски, аномалії |
| `02_etl.ipynb` | ETL-пайплайн: очищення, трансформації, завантаження в PostgreSQL |
| `03_visualizations.ipynb` | Візуалізації: відповіді на аналітичні питання у графіках |
| `04_analysis.ipynb` | Cтатистичне тестування гіпотез |
| `05_conclusions.ipynb` | Зведені висновки та бізнес-рекомендації |